# 01 - FF5 Factor Panel: Static Characterization

**Inertia v2 - Factor Regimes** - Sprint 1

Goal: characterize the five Fama-French factors as standalone return series, *before* we try to time their regimes. This notebook produces:

1. Static performance stats per factor (table).
2. Cumulative growth chart (5 lines, end-of-line value labels).
3. Pairwise correlation matrix.
4. Decade-by-decade Sharpe bar chart.
5. Drawdown small-multiples (5 panels).

All figures and tables are saved to `factor_regimes/figures/` and `factor_regimes/tables/` for downstream consumption (the slide-builder agent reads these directly).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.data import build_factor_panel, factor_static_stats, FF5_FACTORS
from lib.style import (
    apply_style, save_fig, save_table,
    C, FACTOR_PALETTE, FULL_COL,
    bar_value_labels, line_end_labels, legend_below, yearly_xticks,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

FIG_DIR   = '../figures'
TABLE_DIR = '../tables'

## 1. Load panel

In [2]:
ff = build_factor_panel(include_macro=True)
print(f'Panel shape: {ff.shape}')
print(f'Range:       {ff.index.min().date()} -> {ff.index.max().date()}')
ff[FF5_FACTORS].tail()

Panel shape: (752, 30)
Range:       1963-07-31 -> 2026-02-28


,MKT_RF,SMB,HML,RMW,CMA
date,,,,,
2025-10-31,0.0196,-0.0131,-0.0310,-0.0524,-0.0403
2025-11-30,-0.0013,0.0147,0.0376,0.0144,0.0068
2025-12-31,-0.0036,-0.0022,0.0242,0.0040,0.0037
2026-01-31,0.0103,0.0326,0.0372,0.0182,0.0183
2026-02-28,-0.0117,0.0063,0.0283,0.0162,0.0507


## 2. Static performance per factor

In [3]:
stats = factor_static_stats(ff)
save_table(stats, '01_ff5_static_stats', out_dir=TABLE_DIR)
stats

  saved: ../tables/01_ff5_static_stats.{csv,md}


,n_months,start,end,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
MKT_RF,752,1963-07,2026-02,0.0711,0.1543,0.4609,-0.5027,1.7557,-0.5575
SMB,752,1963-07,2026-02,0.0221,0.1048,0.2107,0.3597,3.0732,-0.5647
HML,752,1963-07,2026-02,0.0353,0.1029,0.3430,0.0926,2.1815,-0.5779
RMW,752,1963-07,2026-02,0.0320,0.0769,0.4156,-0.3247,11.1103,-0.4178
CMA,752,1963-07,2026-02,0.0299,0.0717,0.4176,0.2712,1.3640,-0.2725


## 3. Cumulative growth of $1 in each factor (long-only)

Note FF5 factors include long-short legs (SMB, HML, RMW, CMA), so cumulative growth here represents the long-short return stream. Mkt-RF is the equity premium.

In [4]:
cum = (1 + ff[FF5_FACTORS]).cumprod()

fig, ax = plt.subplots(figsize=(FULL_COL, 3.6))
for f in FF5_FACTORS:
    ax.plot(cum.index, cum[f].values, color=FACTOR_PALETTE[f],
            linewidth=1.4, label=f)

ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \\$1 (log)')
ax.set_title('Fama-French 5 factors: cumulative growth, 1963 to 2026',
             loc='left', color=C['ink'])
ax.set_xlim(cum.index.min(), cum.index.max() + pd.DateOffset(months=24))  # padding for labels
yearly_xticks(ax, every=5)

# End-of-line labels (no overlap with chart body)
line_end_labels(ax, cum, colors=FACTOR_PALETTE, fmt='{:.1f}', fontsize=9)

legend_below(ax, ncol=5)
save_fig(fig, '01_ff5_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/01_ff5_cumret.png


## 4. Pairwise correlation matrix

Low correlations between FF5 factors are why we treat them independently in our regime detection.

In [5]:
corr = ff[FF5_FACTORS].corr().round(2)
save_table(corr, '01_ff5_correlation', out_dir=TABLE_DIR)

fig, ax = plt.subplots(figsize=(4.5, 3.8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-0.6, vmax=0.6, aspect='auto')

ax.set_xticks(range(len(FF5_FACTORS)))
ax.set_yticks(range(len(FF5_FACTORS)))
ax.set_xticklabels(FF5_FACTORS, fontsize=9)
ax.set_yticklabels(FF5_FACTORS, fontsize=9)
ax.grid(False)

# Annotate each cell with the correlation value (auto-contrast text)
for i in range(len(FF5_FACTORS)):
    for j in range(len(FF5_FACTORS)):
        v = corr.values[i, j]
        text_color = 'white' if abs(v) > 0.45 else C['ink']
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                fontsize=9, color=text_color)

ax.set_title('FF5 pairwise correlation, 1963 to 2026',
             loc='left', color=C['ink'])
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04, shrink=0.8)
save_fig(fig, '02_ff5_correlation_heatmap', out_dir=FIG_DIR)
plt.show()

  saved: ../tables/01_ff5_correlation.{csv,md}


  saved: ../figures/02_ff5_correlation_heatmap.png


## 5. Decade-by-decade Sharpe ratios

Shows that factor premia are time-varying (the underlying motivation for regime detection).

In [6]:
decades = {
    '1963-1979': ('1963-07', '1979-12'),
    '1980-1999': ('1980-01', '1999-12'),
    '2000-2009': ('2000-01', '2009-12'),
    '2010-2019': ('2010-01', '2019-12'),
    '2020-2026': ('2020-01', None),
}

rows = {}
for label, (lo, hi) in decades.items():
    sub_stats = {}
    for f in FF5_FACTORS:
        r = ff[f].loc[lo:hi].dropna()
        if len(r) < 12:
            sub_stats[f] = np.nan
        else:
            mu = r.mean() * 12
            sd = r.std(ddof=1) * np.sqrt(12)
            sub_stats[f] = mu / sd if sd > 0 else np.nan
    rows[label] = sub_stats

decade_sharpe = pd.DataFrame(rows).T  # rows=decades, cols=factors
save_table(decade_sharpe, '02_ff5_decade_sharpe', out_dir=TABLE_DIR)
decade_sharpe

  saved: ../tables/02_ff5_decade_sharpe.{csv,md}


,MKT_RF,SMB,HML,RMW,CMA
1963-1979,0.1650,0.5791,0.6733,0.0243,0.5021
1980-1999,0.6932,-0.1367,0.3202,0.6481,0.4289
2000-2009,-0.1024,0.5590,0.6219,0.6133,0.8078
2010-2019,1.0146,-0.0475,-0.2985,0.2646,0.0392
2020-2026,0.7091,-0.1831,0.1224,0.5535,0.0800


In [7]:
# Grouped bar chart: x = decade, bars per factor
fig, ax = plt.subplots(figsize=(FULL_COL, 3.8))
n_factors = len(FF5_FACTORS)
bar_w = 0.16
x = np.arange(len(decades))

for i, f in enumerate(FF5_FACTORS):
    pos = x + (i - (n_factors - 1)/2) * bar_w
    bars = ax.bar(pos, decade_sharpe[f].values, bar_w,
                  color=FACTOR_PALETTE[f], label=f, edgecolor='white', linewidth=0.4)

ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(list(decades.keys()), fontsize=9)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('FF5 Sharpe ratios by decade: factor premia are time-varying',
             loc='left', color=C['ink'])
legend_below(ax, ncol=5)
save_fig(fig, '03_ff5_decade_sharpe_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/03_ff5_decade_sharpe_bars.png


## 6. Drawdown small-multiples

Each factor's path of drawdowns since 1963. Some factors (SMB, HML, RMW) experience multi-decade drawdowns. Others (Mkt-RF) recover faster but with deeper troughs.

In [8]:
fig, axes = plt.subplots(5, 1, figsize=(FULL_COL, 6.5), sharex=True)

for ax, f in zip(axes, FF5_FACTORS):
    cum = (1 + ff[f]).cumprod()
    dd = cum / cum.cummax() - 1
    ax.fill_between(dd.index, dd.values, 0,
                    color=FACTOR_PALETTE[f], alpha=0.30, linewidth=0)
    ax.plot(dd.index, dd.values, color=FACTOR_PALETTE[f], linewidth=0.9)
    ax.axhline(0, color=C['muted'], linewidth=0.4)
    mdd = dd.min()
    ax.set_ylim(mdd * 1.15, 0.04)
    ax.set_ylabel(f, color=FACTOR_PALETTE[f], fontsize=10, fontweight='bold')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.text(0.99, 0.10, f'Max DD: {mdd:.0%}',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8.5, color=C['red'])

axes[0].set_title('FF5 drawdowns from trailing peak, 1963 to 2026',
                  loc='left', color=C['ink'])
yearly_xticks(axes[-1], every=10)
save_fig(fig, '04_ff5_drawdown_smallmultiples', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/04_ff5_drawdown_smallmultiples.png


## Takeaways for Sprint 1

- All five FF5 factors have positive long-run mean returns over the 1963 to 2026 window, but with materially different risk profiles.
- Pairwise correlations are mostly low (the largest in absolute value is between HML and CMA). This is what makes a multi-factor regime model interesting: each factor needs its own state-space model.
- Decade Sharpe ratios swing widely. Value (HML) is positive in three decades and negative in two. Profitability (RMW) is consistent. Market (MKT_RF) cycles with the business cycle.
- Each factor experiences multi-year drawdowns. Detecting when a factor is in a drawdown regime is the practical motivation for the next three notebooks (Methods A, B, C).

**Saved artifacts** (consumed by downstream notebooks and the slide-builder agent):

Tables: `01_ff5_static_stats`, `01_ff5_correlation`, `02_ff5_decade_sharpe`

Figures: `01_ff5_cumret`, `02_ff5_correlation_heatmap`, `03_ff5_decade_sharpe_bars`, `04_ff5_drawdown_smallmultiples`